# 08: Scoring

Scores holdout or new data with the saved encoders and model.

In [ ]:
config_path = 'config/car_coll/v1'
score_name = 'holdout'

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import yaml, os, sys, joblib
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment
from preprocessing import condition_data
print("Stage 08: Scoring")
project_root = setup_notebook_environment()

In [ ]:
pc_num = open("current.pc").read().strip()
pc_id = f"PC{pc_num}"

with open(f'{config_path}/config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
output_base = cfg['machines'][pc_id]['paths']['output_path']
score_cfg = cfg['scoring'][score_name]
print(f'Scoring: {score_name}')
print(f'Has target: {score_cfg.get("has_target", False)}')

In [ ]:
# Load data based on source type
print(f'\nLoading data...')

if score_cfg['source'] == 'master':
    # Holdout: master + aux joined, filtered to the holdout folds
    print(f"  Source: master file with fold filter")
    
    aux_path = cfg['machines'][pc_id]['paths']['aux_data_path']
    aux_file = cfg['data']['aux_file']
    fold_col = cfg['data']['fold_column']
    join_key = cfg['data']['join_key']
    
    print(f"  Loading aux: {aux_path}/{aux_file}")
    aux_data = pd.read_parquet(f'{aux_path}/{aux_file}')
    
    needed_folds = score_cfg['folds']
    aux_filtered = aux_data[aux_data[fold_col].isin(needed_folds)]
    print(f"  Filtered aux to folds {needed_folds}: {aux_filtered.shape[0]:,} rows")
    
    master_path = cfg['machines'][pc_id]['paths']['master_data_path']
    master_file = cfg['data']['master_file']
    print(f"  Loading master: {master_path}/{master_file}")
    
    master_data = pd.read_parquet(f'{master_path}/{master_file}')
    print(f"  Loaded master: {master_data.shape}")
    
    aux_cols = [join_key, fold_col]
    if score_cfg.get('has_target', False):
        target = cfg['experiment']['target']
        if target in aux_filtered.columns:
            aux_cols.append(target)
            print(f'  Including target from aux: {target}')
        # Source columns for combined target (e.g. pp_bi, pp_pd); exposures come from master
        from utils import combine_source_columns
        aux_cols += [c for c in combine_source_columns(cfg) if c in aux_filtered.columns and c not in aux_cols and c not in master_data.columns]
    # Columns training (Stage 01) took from aux, not master -> use aux values here too
    from utils import aux_feature_columns
    aux_feats = aux_feature_columns(cfg, config_path, master_data.columns, aux_filtered.columns)
    if aux_feats:
        print(f'  Using aux values (as in training): {aux_feats}')
        master_data = master_data.drop(columns=aux_feats)
        aux_cols += [c for c in aux_feats if c not in aux_cols]
    df = master_data.merge(aux_filtered[aux_cols], on=join_key, how='inner')
    from utils import add_combined_columns
    df = add_combined_columns(df, cfg)
    print(f"  After join: {df.shape}")
    print(f"  Fold distribution: {df[fold_col].value_counts().sort_index().to_dict()}")
    
elif score_cfg['source'] == 'file':
    # New data: read the file as is
    print(f"  Source: file (new data)")
    data_path = score_cfg.get('data_dir', cfg['machines'][pc_id]['paths']['master_data_path'])
    file_name = score_cfg['file']
    print(f"  Loading: {data_path}/{file_name}")
    df = pd.read_parquet(f'{data_path}/{file_name}')
    print(f"  Loaded: {df.shape}")
    
else:
    raise ValueError(f"Unknown source type: {score_cfg['source']}")

print(f'\nData loaded: {df.shape}')

# Save target and exposure for metrics (if available)
target_values = None
exposure_values = None
if score_cfg.get('has_target', False):
    target = cfg['experiment']['target']
    exposure = cfg['experiment']['exposure']
    if target in df.columns:
        target_values = df[target].copy()
        print(f'  Saved target for metrics: {target}')
    if exposure in df.columns:
        exposure_values = df[exposure].copy()
        print(f'  Saved exposure for metrics: {exposure}')

In [ ]:
# Apply loss cap and exposure floor (match training pipeline)
print(f'\nApplying loss cap and exposure floor...')
target = cfg['experiment']['target']
exposure = cfg['experiment']['exposure']
loss_cap = cfg['experiment'].get('loss_cap', None)
loss_type = cfg['experiment'].get('loss_type', 'bi')
exposure_floor = cfg['experiment'].get('exposure_floor', None)

if score_cfg.get('has_target', False) and target in df.columns:
    from model_utils import apply_loss_cap_single
    df = apply_loss_cap_single(
        df,
        loss_cap=loss_cap,
        target_col=target,
        exposure_col=exposure,
        exposure_floor=exposure_floor,
        verbose=True
    )
    # Metrics/charts use the same capped target and floored exposure as training
    target_values_uncapped = (target_values * exposure_values).loc[df.index] / df[exposure]
    target_values = df[target].copy()
    exposure_values = df[exposure].copy()
else:
    print(f'  Skipping loss cap (no target in data)')
    if exposure_floor and exposure in df.columns:
        import numpy as np
        n_floor = (df[exposure] < exposure_floor).sum()
        df[exposure] = np.maximum(df[exposure], exposure_floor)
        if n_floor > 0:
            print(f'  Exposure floor ({exposure_floor}): {n_floor:,} records adjusted')

print(f'  Shape after loss cap/floor: {df.shape}')

In [ ]:
# Preserve target and exposure before conditioning
print(f'\nConditioning...')
target = cfg['experiment']['target']
exposure = cfg['experiment']['exposure']

target_data = None
exposure_data = None
if target in df.columns:
    target_data = df[target].copy()
    print(f'  Preserved target: {target}')
if exposure in df.columns:
    exposure_data = df[exposure].copy()
    print(f'  Preserved exposure: {exposure}')

# Apply conditioning (this may drop target/exposure)
df = condition_data(df, cfg, type_conversions_path=f'{config_path}/type_conversions.csv')
print(f'  After conditioning: {df.shape}')

if target_data is not None:
    df[target] = target_data
    print(f'  Restored target: {target}')
if exposure_data is not None:
    df[exposure] = exposure_data
    print(f'  Restored exposure: {exposure}')

print(f'  Final shape: {df.shape}')

In [ ]:
# Drop auxiliary columns before encoding
print(f'\nDropping auxiliary columns...')
aux_cols = ['fold']  # keep vin_date for the GLM join
cols_to_drop = [c for c in aux_cols if c in df.columns]
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f'  Dropped: {cols_to_drop}')
else:
    print(f'  No aux columns to drop')
print(f'  Data shape: {df.shape}')

In [ ]:
print(f'\nLoading encoder...')
import joblib
with open(f'{output_base}/models/04c_encoders.pkl', 'rb') as f:
    encoders = joblib.load(f)

encoding_summary = pd.read_csv(f'{output_base}/results/04c_encoding_summary.csv')
print(f'  Loaded {len(encoders)} encoders and encoding summary')

# Encode features
print(f'\nEncoding...')
from feature_encoder import apply_saved_encoders
df_encoded = apply_saved_encoders(df, encoders, encoding_summary, config_path=config_path)
print(f'  Encoded: {df_encoded.shape}')

# Compare with training columns and fix mismatch
X_train_ref = pd.read_parquet(f'{output_base}/data/04c_train_encoded.parquet')
train_cols = set(X_train_ref.columns)
encoded_cols = set(df_encoded.columns)
extra_in_encoded = encoded_cols - train_cols
missing_in_encoded = train_cols - encoded_cols
if extra_in_encoded:
    print(f'  [!] EXTRA columns: {extra_in_encoded}')
    df_encoded = df_encoded.drop(columns=list(extra_in_encoded))
    print(f'  Dropped extra. New shape: {df_encoded.shape}')
if missing_in_encoded:
    print(f'  [!] MISSING columns: {missing_in_encoded}')

In [ ]:
# Load model
print(f'\nLoading model...')
model_file = f'{output_base}/models/05c_model.json'
model = xgb.XGBRegressor()
model.load_model(model_file)
print(f'  Model loaded: {model_file}')

# Validate features match
from model_validation import validate_features_match
validate_features_match(model, df_encoded, 'df_encoded')

# Get target method and predict
target_method = cfg.get('model', {}).get('target_method', 'direct')
print(f'\nPredicting (method={target_method})...')
if target_method == 'residual':
    from model_utils import load_glm_for_scoring, predict_with_transform
    glm_preds, matched_idx = load_glm_for_scoring(df, cfg, score_cfg, pc_id, target_method)
    
    # Filter data to rows with GLM predictions
    if matched_idx is not None:
        df = df.loc[matched_idx]
        df_encoded = df_encoded.loc[matched_idx]
    
    _, predictions = predict_with_transform(model, df_encoded, glm_preds, target_method)
    if predictions.isna().any():
        print(f"  Filtering {predictions.isna().sum()} NaN predictions")
        valid_mask = ~predictions.isna()
        predictions = predictions[valid_mask]
    df = df.loc[predictions.index].copy()
    df['prediction'] = predictions.values
    df['glm_pred'] = glm_preds.loc[df.index].values
    
    if target_values is not None:
        target_values = target_values.loc[df.index]
    if exposure_values is not None:
        exposure_values = exposure_values.loc[df.index]
else:
    df['prediction'] = model.predict(df_encoded)
print(f'  Mean: {df["prediction"].mean():.2f}')

# Validate predictions
from model_validation import validate_predictions
validate_predictions(df['prediction'].values, df)

In [ ]:
print(f'\nSaving...')
output_dir = f'{output_base}/scoring/{score_name}'
os.makedirs(output_dir, exist_ok=True)
# vin_date back to a column
from utils import save_parquet
save_parquet(df.reset_index(), f'{output_dir}/predictions.parquet', index=False)
print(f'  Saved: scoring/{score_name}/predictions.parquet')

In [ ]:
# Calculate metrics (if target available)
if score_cfg.get('has_target', False):
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    import numpy as np
    import pandas as pd
    
    if target_values is None or exposure_values is None:
        print('\n[!] Warning: Target/exposure not available for metrics')
    else:
        print(f'\nCalculating metrics...')
        
        metrics_df = pd.DataFrame({
            'target': target_values,
            'prediction': df['prediction'],
            'exposure': exposure_values
        })
        
        print(f'  Total rows: {len(metrics_df):,}')
        
        nan_mask = metrics_df.isna().any(axis=1)
        n_nan = nan_mask.sum()
        if n_nan > 0:
            print(f'  [!] Found {n_nan:,} rows with NaN values')
            print(f'    NaN in target: {metrics_df["target"].isna().sum():,}')
            print(f'    NaN in prediction: {metrics_df["prediction"].isna().sum():,}')
            print(f'    NaN in exposure: {metrics_df["exposure"].isna().sum():,}')
            print(f'  Filtering out NaN rows...')
            metrics_df = metrics_df[~nan_mask]
        
        print(f'  Valid rows for metrics: {len(metrics_df):,}')
        
        mae = mean_absolute_error(
            metrics_df['target'],
            metrics_df['prediction'],
            sample_weight=metrics_df['exposure']
        )
        rmse = np.sqrt(mean_squared_error(
            metrics_df['target'],
            metrics_df['prediction'],
            sample_weight=metrics_df['exposure']
        ))
        
        print(f'  MAE: {mae:.4f}')
        print(f'  RMSE: {rmse:.4f}')
        
        # Save metrics
        metrics = {
            'score_name': score_name,
            'mae': float(mae),
            'rmse': float(rmse),
            'n_rows': len(metrics_df),
            'n_rows_with_nan': int(n_nan) if n_nan > 0 else 0
        }
        
        with open(f'{output_dir}/metrics.yaml', 'w') as f:
            yaml.dump(metrics, f)
        print(f'  Metrics saved to {output_dir}/metrics.yaml')
else:
    print('\nNo target available (has_target=False)')

In [ ]:
# Loss cap / exposure floor already applied in cell 5; align uncapped target to scored rows
if score_cfg.get('has_target', False) and target_values is not None:
    target_values_uncapped = target_values_uncapped.loc[target_values.index]
else:
    target_values_uncapped = None


In [ ]:
# Create lift chart (if target available)
if score_cfg.get('has_target', False) and target_values is not None:
    from lift_chart_fast import generate_scoring_lift_chart
    
    target = cfg['experiment']['target']
    exposure = cfg['experiment']['exposure']
    
    table = generate_scoring_lift_chart(
        df['prediction'],
        target_values,
        exposure_values,
        exposure,
        output_dir,
        output_base,
        score_name=score_name
    )
else:
    print(f'\\n* Skipping lift chart (no target available)')

In [ ]:
# Ratio lift chart (residual method only)
from lift_chart_fast import generate_scoring_ratio_lift_chart
if score_cfg.get('has_target', False) and target_values is not None and 'glm_pred' in df.columns:
    table_r = generate_scoring_ratio_lift_chart(df['prediction'], target_values, df['glm_pred'], exposure_values,
                                                exposure, output_dir, output_base, score_name=score_name)
else:
    print('\nSkipping ratio chart (no target or not residual method)')

In [ ]:
# Calculate actuarial metrics (if target available)
if score_cfg.get('has_target', False) and target_values is not None:
    from hpo_metrics import compute_fit_quality, compute_model_power
    
    print(f'\nCalculating actuarial metrics...')
    
    if 'metrics_df' in locals():
        clean_target = metrics_df['target'].values
        clean_pred = metrics_df['prediction'].values
        clean_exposure = metrics_df['exposure'].values
    else:
        temp_df = pd.DataFrame({
            'target': target_values,
            'prediction': df['prediction'],
            'exposure': exposure_values
        })
        temp_df = temp_df.dropna()
        clean_target = temp_df['target'].values
        clean_pred = temp_df['prediction'].values
        clean_exposure = temp_df['exposure'].values
    
    fit_quality = compute_fit_quality(clean_target, clean_pred, clean_exposure)
    model_power = compute_model_power(clean_target, clean_pred, clean_exposure)
    
    print(f'\nActuarial Metrics ({score_name}):')
    print(f'    Fit Quality:  {fit_quality:.4f}')
    print(f'    Model Power:  {model_power:.4f}')
    
    if 'metrics' in locals():
        metrics['fit_quality'] = float(fit_quality)
        metrics['model_power'] = float(model_power)
        with open(f'{output_dir}/metrics.yaml', 'w') as f:
            yaml.dump(metrics, f)
        print(f'  Updated metrics.yaml')
else:
    print(f'\nSkipping actuarial metrics (no target available)')

In [ ]:
# Uncapped lift: predictions vs uncapped actual losses
if score_cfg.get('has_target', False) and target_values_uncapped is not None:
    from lift_chart_fast import generate_scoring_lift_chart
    
    target = cfg['experiment']['target']
    exposure = cfg['experiment']['exposure']
    
    print(f'\nCreating UNCAPPED lift chart...')
    print(f'  This shows model performance against actual (uncapped) losses')
    
    table_uncapped = generate_scoring_lift_chart(
        df['prediction'],
        target_values_uncapped,
        exposure_values,
        exposure,
        output_dir,
        output_base,
        score_name=score_name,
        suffix='_uncapped'
    )
else:
    print(f'\nSkipping uncapped lift chart (no target available)')